## Тест двух способов запроса данных gharchive из Google BigQuery
https://www.gharchive.org/

In [3]:
import pandas as pd
from google.oauth2 import service_account
from google.cloud import bigquery
import time


JSON_ACCOUNT_PATH = "auth/inner-suprstate-406414-cc3214360b12.json"

# Формируем запрос и получаем количество вопросов с тегом "pandas", сгруппированные по дате создания
# query = '''
# SELECT 
#   * 
# FROM `githubarchive.day.201901`
# WHERE 
#   type = 'PushEvent'
# LIMIT 1000000
# '''
query = """
SELECT 
  COUNT(*) 
FROM `githubarchive.day.2015*`
WHERE 
  type = 'PushEvent'
  AND (_TABLE_SUFFIX BETWEEN '0101' AND '0105')
"""

# Указываем идентификатор проекта
project_id = "inner-suprstate-406414"

# 1 способ (обертка над API Google BigQuery https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_gbq.html)

# Прописываем адрес к файлу с данными по сервисному аккаунту и получаем credentials для доступа к данным
credentials = service_account.Credentials.from_service_account_file(
    JSON_ACCOUNT_PATH
)
start_time = time.time()

df = pd.read_gbq(query, project_id=project_id, credentials=credentials)

end_time = time.time()
print(f"1 способ: {end_time - start_time} секунд")


# 2 способ (официальная библиотека)
client = bigquery.Client.from_service_account_json(
    JSON_ACCOUNT_PATH
)

start_time = time.time()

df2 = client.query(query, project=project_id).to_dataframe()

end_time = time.time()
print(f"2 способ: {end_time - start_time} секунд")

1 способ: 2.6346311569213867 секунд
2 способ: 3.8251638412475586 секунд


In [4]:
query = """
SELECT 
  * 
FROM `githubarchive.day.20190701`
WHERE 
  type = 'PushEvent'
LIMIT 10000
"""

start_time = time.time()

df = pd.read_gbq(query, project_id=project_id, credentials=credentials)

end_time = time.time()
print(f"1 способ: {end_time - start_time} секунд")


start_time = time.time()

df2 = client.query(query, project=project_id).to_dataframe()

end_time = time.time()
print(f"2 способ: {end_time - start_time} секунд")

1 способ: 13.901902437210083 секунд
2 способ: 4.4870734214782715 секунд


## Второй тест ближе к условиям реальной задачи, значит будем использовать официальную библиотеку
## Также в официальной библиотеке доступно больше функций
https://www.datalytics.ru/all/kak-ispolzovat-google-bigquery-s-pomoschyu-python/